In [ ]:
import pandas as pd
import numpy as np

# --- Load dataset ---
df = pd.read_csv("Nat_Gas.csv")

# Convert date format properly
df['Dates'] = pd.to_datetime(df['Dates'], format='%m/%d/%y')

# Sort and set index
df = df.sort_values('Dates')
df.set_index('Dates', inplace=True)

# Rename for consistency
df.rename(columns={'Prices': 'Price'}, inplace=True)

# --- Step 1: Convert monthly → daily using interpolation ---
daily_df = df.resample('D').interpolate(method='linear')

# --- Step 2: Extract seasonality ---
daily_df['month'] = daily_df.index.month
monthly_avg = daily_df.groupby('month')['Price'].mean()

# --- Step 3: Build trend model ---
daily_df['time'] = np.arange(len(daily_df))
coeffs = np.polyfit(daily_df['time'], daily_df['Price'], 1)

def trend(t):
    return coeffs[0]*t + coeffs[1]

# --- Step 4: Forecast next 1 year ---
future_dates = pd.date_range(start=daily_df.index[-1], periods=365, freq='D')
future_time = np.arange(len(daily_df), len(daily_df) + 365)

future_prices = []

for i, date in enumerate(future_dates):
    month = date.month
    seasonal_adjustment = monthly_avg[month] - monthly_avg.mean()
    predicted_price = trend(future_time[i]) + seasonal_adjustment
    future_prices.append(predicted_price)

future_df = pd.DataFrame({'Price': future_prices}, index=future_dates)

# --- Combine past + future ---
full_df = pd.concat([daily_df[['Price']], future_df])

# --- Final function ---
def estimate_price(input_date):
    input_date = pd.to_datetime(input_date)

    if input_date in full_df.index:
        return float(full_df.loc[input_date]['Price'])
    else:
        # interpolate for exact date
        temp = full_df.reindex(full_df.index.union([input_date])).sort_index()
        temp = temp.interpolate()
        return float(temp.loc[input_date]['Price'])

# Example
print(estimate_price("2025-06-15"))

In [ ]:
import pandas as pd
# --- Function ---
def price_storage_contract(
    injection_dates,
    withdrawal_dates,
    volume_per_trade,
    max_storage,
    storage_cost_per_month,
    injection_cost,
    withdrawal_cost,
    transport_cost
):
    
    total_profit = 0
    current_storage = 0
    
    for inj_date, wd_date in zip(injection_dates, withdrawal_dates):
        
        # --- Get prices ---
        buy_price = estimate_price(inj_date)
        sell_price = estimate_price(wd_date)
        
        # --- Check storage limit ---
        if current_storage + volume_per_trade > max_storage:
            print("Storage limit exceeded")
            break
        
        # --- Buy ---
        buy_cost = buy_price * volume_per_trade
        
        # --- Store ---
        months = (pd.to_datetime(wd_date) - pd.to_datetime(inj_date)).days / 30
        storage_cost = months * storage_cost_per_month
        
        # --- Sell ---
        sell_revenue = sell_price * volume_per_trade
        
        # --- Other costs ---
        total_extra_cost = injection_cost + withdrawal_cost + 2 * transport_cost
        
        # --- Profit per cycle ---
        profit = sell_revenue - buy_cost - storage_cost - total_extra_cost
        
        total_profit += profit
        current_storage += volume_per_trade
    
    return total_profit


# --- TEST ---
profit = price_storage_contract(
    injection_dates=["2025-05-01", "2025-06-01"],
    withdrawal_dates=["2025-12-01", "2026-01-01"],
    volume_per_trade=1_000_000,
    max_storage=2_000_000,
    storage_cost_per_month=100000,
    injection_cost=10000,
    withdrawal_cost=10000,
    transport_cost=50000
)

print("Estimated Contract Value:", profit)